# Fraud / Anomaly Detection
## Notebook 04 — Feature Engineering

### Objective

Create meaningful features from transaction time and amount while preserving the original anonymized features.

### Important Principle

The `Class` column represents the known fraud label. It will not be used to create model features because this is an unsupervised anomaly detection project.

The label will be retained separately for later evaluation.

In [3]:
import pandas as pd
import numpy as np

In [5]:
df = pd.read_csv("../data/processed/creditcard_cleaned.csv")

In [7]:
df.shape

(283726, 31)

In [9]:
y = df["Class"].copy()

In [11]:
X = df.drop(columns=["Class"]).copy()

In [13]:
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (283726, 30)
y shape: (283726,)


In [15]:
X.columns.tolist()

['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount']

In [17]:
X_engineered = X.copy()

In [19]:
X_engineered["Amount_log"] = np.log1p(X_engineered["Amount"])

In [21]:
X_engineered["Time_hours"] = X_engineered["Time"] / 3600

In [23]:
X_engineered["Time_days"] = X_engineered["Time"] / (3600 * 24)

In [25]:
X_engineered[
    ["Time", "Time_hours", "Time_days"]
].head()

,Time,Time_hours,Time_days
0,0.0,0.000000,0.000000
1,0.0,0.000000,0.000000
2,1.0,0.000278,0.000012
3,1.0,0.000278,0.000012
4,2.0,0.000556,0.000023


In [27]:
X_engineered["Hour_of_day"] = (
    X_engineered["Time"] / 3600
) % 24

In [29]:
X_engineered["Hour_of_day"].describe()

count    283726.000000
mean         14.537139
std           5.846094
min           0.000000
25%          10.597500
50%          15.008889
75%          19.329167
max          23.999444
Name: Hour_of_day, dtype: float64

In [31]:
X_engineered["Hour_sin"] = np.sin(
    2 * np.pi * X_engineered["Hour_of_day"] / 24
)

X_engineered["Hour_cos"] = np.cos(
    2 * np.pi * X_engineered["Hour_of_day"] / 24
)

In [33]:
amount_mean = X_engineered["Amount"].mean()
amount_std = X_engineered["Amount"].std()

print("Mean amount:", amount_mean)
print("Standard deviation:", amount_std)

Mean amount: 88.47268731099724
Standard deviation: 250.39943711580162


In [35]:
X_engineered["Amount_zscore"] = (
    X_engineered["Amount"] - amount_mean
) / amount_std

In [37]:
X_engineered["Amount_zscore"].describe()

count    2.837260e+05
mean    -5.749935e-17
std      1.000000e+00
min     -3.533262e-01
25%     -3.309620e-01
50%     -2.654666e-01
75%     -4.378080e-02
max      1.022474e+02
Name: Amount_zscore, dtype: float64

In [39]:
amount_median = X_engineered["Amount"].median()

amount_mad = np.median(
    np.abs(
        X_engineered["Amount"] - amount_median
    )
)

print("Median:", amount_median)
print("MAD:", amount_mad)

Median: 22.0
MAD: 20.22


In [41]:
X_engineered["Amount_robust_deviation"] = (
    X_engineered["Amount"] - amount_median
) / (amount_mad + 1e-9)

In [43]:
X_engineered["Amount_percentile"] = (
    X_engineered["Amount"]
    .rank(pct=True)
)

In [45]:
X_engineered[
    [
        "Amount",
        "Amount_log",
        "Amount_zscore",
        "Amount_robust_deviation",
        "Amount_percentile"
    ]
].head()

,Amount,Amount_log,Amount_zscore,Amount_robust_deviation,Amount_percentile
0,149.62,5.014760,0.244199,6.311573,0.859116
1,2.69,1.305626,-0.342583,-0.954995,0.185131
2,378.66,5.939276,1.158898,17.638971,0.952038
3,123.50,4.824306,0.139886,5.019782,0.831998
4,69.99,4.262539,-0.073813,2.373393,0.730608


In [47]:
X_engineered.shape

(283726, 39)

In [49]:
X_engineered.isnull().sum().sum()

0

In [51]:
np.isinf(X_engineered).sum().sum()

0

In [53]:
X_engineered.describe().T

,count,mean,std,min,25%,50%,75%,max
Time,283726.0,9.481108e+04,47481.047891,0.000000,54204.750000,84692.500000,139298.000000,172792.000000
V1,283726.0,5.917150e-03,1.948026,-56.407510,-0.915951,0.020384,1.316068,2.454930
V2,283726.0,-4.134756e-03,1.646703,-72.715728,-0.600321,0.063949,0.800283,22.057729
V3,283726.0,1.613119e-03,1.508682,-48.325589,-0.889682,0.179963,1.026960,9.382558
V4,283726.0,-2.966308e-03,1.414184,-5.683171,-0.850134,-0.022248,0.739647,16.875344
V5,283726.0,1.827560e-03,1.377008,-113.743307,-0.689830,-0.053468,0.612218,34.801666
V6,283726.0,-1.139488e-03,1.331931,-26.160506,-0.769031,-0.275168,0.396792,73.301626
V7,283726.0,1.800692e-03,1.227664,-43.557242,-0.552509,0.040859,0.570474,120.589494
V8,283726.0,-8.544526e-04,1.179054,-73.216718,-0.208828,0.021898,0.325704,20.007208
V9,283726.0,-1.596200e-03,1.095492,-13.434066,-0.644221,-0.052596,0.595977,15.594995


In [55]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [57]:
X_engineered.to_csv(
    "../data/processed/creditcard_engineered_features.csv",
    index=False
)

In [59]:
y.to_csv(
    "../data/processed/fraud_labels.csv",
    index=False
)

In [61]:
import os

print(
    "Engineered features:",
    os.path.exists(
        "../data/processed/creditcard_engineered_features.csv"
    )
)

print(
    "Fraud labels:",
    os.path.exists(
        "../data/processed/fraud_labels.csv"
    )
)

Engineered features: True
Fraud labels: True


# Feature Engineering Summary

## Features Created

### Amount Features
- Amount_log
- Amount_zscore
- Amount_robust_deviation
- Amount_percentile

### Time Features
- Time_hours
- Time_days
- Hour_of_day
- Hour_sin
- Hour_cos

## Label Handling

The original `Class` column was separated from the model features.

- `X_engineered` → model input
- `y` → evaluation only

The fraud label will NOT be provided to the unsupervised anomaly detection algorithms.

## Output Files

- `creditcard_engineered_features.csv`
- `fraud_labels.csv`

These files will be used in the preprocessing and anomaly detection notebooks.

In [63]:
X_engineered.shape

(283726, 39)

In [65]:
X_engineered.isnull().sum().sum()

0

In [67]:
np.isinf(X_engineered).sum().sum()

0

In [69]:
X_engineered.columns.tolist()

['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount',
 'Amount_log',
 'Time_hours',
 'Time_days',
 'Hour_of_day',
 'Hour_sin',
 'Hour_cos',
 'Amount_zscore',
 'Amount_robust_deviation',
 'Amount_percentile']